In [1]:
import joblib
import numpy as np
import pandas as pd

model_package = joblib.load(
    "capstone_outputs/respiratory_capstone_model.joblib"
)

model = model_package["model"]

print("Model loaded successfully!")
print("Best model:", model_package["best_model"])

Model loaded successfully!
Best model: Random forest


In [2]:
import re
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.metrics import (
    roc_auc_score, average_precision_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score, brier_score_loss,
    classification_report, ConfusionMatrixDisplay, RocCurveDisplay
)
import joblib

In [4]:
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

In [5]:
# 1. Load the Excel file
DATA_FILE = Path("Health Unit Raw Data_complete.xlsx")
OUTPUT_DIR = Path("capstone_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

In [6]:
# 1. Load the Excel file
DATA_FILE = Path("Health Unit Raw Data_complete.xlsx")
OUTPUT_DIR = Path("capstone_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

raw = pd.read_excel(DATA_FILE)


In [7]:
raw.columns = [
    str(c).strip().lower().replace(".", "").replace(" ", "_")
    for c in raw.columns
]

In [8]:
print("Original shape:", raw.shape)
print("Columns:", raw.columns.tolist())

Original shape: (5339, 5)
Columns: ['registration_number', 'age', 'gender', 'month', 'disease']


In [9]:
# Keep only columns available in the dataset
raw.columns = (
    raw.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(".", "", regex=False)
    .str.replace(" ", "_", regex=False)
)

raw = raw.rename(columns={"registration_number": "reg_no"})

required = ["reg_no", "age", "gender", "month", "disease"]
df = raw[required].copy()

print(df.columns.tolist())

['reg_no', 'age', 'gender', 'month', 'disease']


In [10]:
# 2. Clean variables
# Registration number is only used to prevent repeat-student leakage.
df["reg_no"] = df["reg_no"].astype("string").str.strip()

In [11]:
# Convert invalid age values such as A or `A to missing
# The dataset is university health-unit data, so values outside 16-30 are treated as invalid.
df["age"] = pd.to_numeric(df["age"], errors="coerce")
df.loc[~df["age"].between(16, 30), "age"] = np.nan


In [12]:
# Standardize gender
df["gender"] = df["gender"].astype("string").str.upper().str.strip()
df.loc[~df["gender"].isin(["M", "F"]), "gender"] = pd.NA

In [13]:
# Keep valid calendar months only
df["month"] = pd.to_numeric(df["month"], errors="coerce")
df.loc[~df["month"].between(1, 12), "month"] = np.nan


In [14]:
# Clean disease text
df["disease_clean"] = (
    df["disease"].fillna("").astype(str).str.lower()
    .str.replace(r"[^a-z0-9]+", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [15]:
# Disease text and month are required to create the target
model_df = df[df["disease_clean"].ne("") & df["month"].notna()].copy()

In [16]:
print("Usable rows:", len(model_df))
print("Missing values:\n", model_df[["age", "gender", "month"]].isna().sum())

Usable rows: 5329
Missing values:
 age       575
gender      0
month       0
dtype: int64


In [17]:
median_age = model_df["age"].median()
model_df["age"] = model_df["age"].fillna(median_age)

print("Median age used:", median_age)
print("Missing ages after replacement:", model_df["age"].isna().sum())

Median age used: 20.0
Missing ages after replacement: 0


In [18]:
# 3. Create the respiratory target
# Review this keyword definition with a clinician before using the model operationally.
respiratory_terms = [
    r"\brti\b", r"\burti\b", r"respiratory", r"\bflu\b", r"influenza",
    r"common cold", r"\bcold\b", r"cough", r"running nose", r"runningnose",
    r"blocked nose", r"sore throat", r"tonsil", r"pneumonia", r"bronch",
    r"asthma", r"chest infection"
]

In [19]:
respiratory_regex = "|".join(respiratory_terms)

In [20]:
model_df["respiratory"] = model_df["disease_clean"].str.contains(
    respiratory_regex, regex=True
).astype(int)

In [21]:
# 4. Engineer seasonal features
# Sine/cosine preserves the fact that December and January are adjacent seasons.
model_df["month_sin"] = np.sin(2 * np.pi * model_df["month"] / 12)
model_df["month_cos"] = np.cos(2 * np.pi * model_df["month"] / 12)

In [22]:
# Group repeated visits by registration number during validation.
# Missing registration numbers receive unique groups.
model_df["student_group"] = model_df["reg_no"].fillna(
    pd.Series(
        ["missing_" + str(i) for i in model_df.index],
        index=model_df.index
    )
).astype(str)

In [23]:
X = model_df[["age", "gender", "month_sin", "month_cos"]]
y = model_df["respiratory"]
groups = model_df["student_group"]

In [24]:
print("Respiratory outcome counts:\n", y.value_counts())
print("Respiratory visit share:", round(y.mean(), 3))

Respiratory outcome counts:
 respiratory
0    3636
1    1693
Name: count, dtype: int64
Respiratory visit share: 0.318


In [25]:
print("\nMonthly summary:\n", month_summary)
month_summary.to_csv(OUTPUT_DIR / "monthly_summary.csv", index=False)

NameError: name 'month_summary' is not defined

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=month_summary, x="month", y="respiratory_share", color="#377eb8")
plt.title("Respiratory share of health-unit visits by month")
plt.xlabel("Month")
plt.ylabel("Respiratory visit share")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
###RTI PERCENTAGE BY AGE
age_rti = (
    model_df
    .groupby("age")["respiratory"]
    .mean()
    .reset_index()
)

age_rti["rti_percentage"] = age_rti["respiratory"] * 100

plt.figure(figsize=(10, 5))

sns.barplot(
    data=age_rti,
    x="age",
    y="rti_percentage",
    errorbar=None
)

plt.title("Percentage of Respiratory-Related Visits by Age")
plt.xlabel("Age (years)")
plt.ylabel("Respiratory-related visits (%)")

plt.tight_layout()
plt.show()

In [ ]:
count
mean
std
min
25%
50%
75%
max

In [ ]:
### RTI BY GENDER
gender_rti = (
    model_df
    .groupby("gender")["respiratory"]
    .mean()
    .reset_index()
)

gender_rti["rti_percentage"] = gender_rti["respiratory"] * 100

print(gender_rti)

plt.figure(figsize=(7, 5))

ax = sns.barplot(
    data=gender_rti,
    x="gender",
    y="rti_percentage",
    errorbar=None
)

plt.title("Percentage of Respiratory-Related Visits by Gender")
plt.xlabel("Gender")
plt.ylabel("Respiratory-related visits (%)")

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.1f%%",
        padding=3
    )

plt.tight_layout()
plt.show()

In [ ]:
# 6. Compare three models
preprocessor = ColumnTransformer([
    (
        "numeric",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]),
        ["age", "month_sin", "month_cos"]
    ),
    (
        "categorical",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]),
        ["gender"]
    )
])

In [ ]:
models = {
    "Logistic regression": LogisticRegression(
        max_iter=3000, class_weight="balanced", random_state=42
    ),
    "Random forest": RandomForestClassifier(
        n_estimators=600,
        min_samples_leaf=15,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    "Gradient boosting": HistGradientBoostingClassifier(
        max_iter=250,
        learning_rate=0.04,
        max_leaf_nodes=12,
        min_samples_leaf=25,
        l2_regularization=1.0,
        random_state=42
    )
}


In [ ]:
# Grouped CV prevents the same student appearing in both training and validation.
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)


In [ ]:
results = []
probabilities = {}

for name, classifier in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", classifier)
    ])

    probabilities_cv = cross_val_predict(
        pipeline, X, y,
        groups=groups,
        cv=cv,
        method="predict_proba",
        n_jobs=-1
    )[:, 1]

    predictions = (probabilities_cv >= 0.50).astype(int)

    results.append({
        "model": name,
        "roc_auc": roc_auc_score(y, probabilities_cv),
        "pr_auc": average_precision_score(y, probabilities_cv),
        "balanced_accuracy": balanced_accuracy_score(y, predictions),
        "precision": precision_score(y, predictions, zero_division=0),
        "recall": recall_score(y, predictions, zero_division=0),
        "f1": f1_score(y, predictions, zero_division=0),
        "brier_score": brier_score_loss(y, probabilities_cv)
    })

comparison = pd.DataFrame(results).sort_values("f1", ascending=False)
print(comparison.round(3))

In [ ]:
# STEP 2: Train and save the Random Forest model

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
import joblib

# Create the Random Forest pipeline
random_forest_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=600,
        min_samples_leaf=15,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

# Train Random Forest using all available data
random_forest_pipeline.fit(X, y)

# Save the Random Forest model
joblib.dump(
    {
        "model": random_forest_pipeline,
        "target": "Respiratory-related health-unit visit",
        "features": ["age", "gender", "month_sin", "month_cos"],
        "best_model": "Random Forest",
        "metrics": comparison.to_dict("records")
    },
    OUTPUT_DIR / "respiratory_random_forest_model.joblib"
)

print("Random Forest trained successfully!")
print("Saved to:", OUTPUT_DIR / "respiratory_random_forest_model.joblib")

In [ ]:
#######Gradient boosting: highest ROC-AUC (0.604), but detects only 7.4% of respiratory cases, so it’s poor at the default 0.50 cutoff.
###Random forest: nearly the same ROC-AUC (0.603) but much better recall (57.4%) and F1 (0.465), making it the best practical choice.
###Logistic regression: weakest overall

In [ ]:
import sys
import sklearn
import numpy
import joblib

print("Python:", sys.version)
print("scikit-learn:", sklearn.__version__)
print("numpy:", numpy.__version__)
print("joblib:", joblib.__version__)

In [ ]:
# 7. Plot ROC curves
plt.figure(figsize=(8, 6))
for name, probability in probabilities.items():
    RocCurveDisplay.from_predictions(y, probability, name=name)
plt.plot([0, 1], [0, 1], "k--", alpha=0.6)
plt.title("Grouped 5-fold cross-validated ROC curves")
plt.tight_layout()
plt.show()

In [ ]:
# 8. Train and save the best model on all available rows
best_name = comparison.iloc[0]["model"]
best_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", models[best_name])
])
best_pipeline.fit(X, y)

In [ ]:
joblib.dump(
    {
        "model": best_pipeline,
        "target": "Respiratory-related health-unit visit",
        "features": ["age", "gender", "month_sin", "month_cos"],
        "best_model": best_name,
        "metrics": comparison.to_dict("records")
    },
    OUTPUT_DIR / "respiratory_capstone_model.joblib"
)
print("Saved model:", OUTPUT_DIR / "respiratory_capstone_model.joblib")


In [ ]:
# 9. Predict new example records
new_students = pd.DataFrame({
    "age": [20, 22, 19],
    "gender": ["F", "M", "F"],
    "month": [4, 10, 11]
})

In [ ]:
new_students["month_sin"] = np.sin(2 * np.pi * new_students["month"] / 12)
new_students["month_cos"] = np.cos(2 * np.pi * new_students["month"] / 12)

In [ ]:
new_students["respiratory_probability"] = best_pipeline.predict_proba(
    new_students[["age", "gender", "month_sin", "month_cos"]]
)[:, 1]
new_students["predicted_class"] = np.where(
    new_students["respiratory_probability"] >= 0.50,
    "respiratory-related",
    "other"
)
print("\nExample predictions:\n", new_students)
new_students.to_csv(OUTPUT_DIR / "example_predictions.csv", index=False)

In [ ]:
def predict_respiratory(age, gender, month):
    # Calculate seasonal features
    month_sin = np.sin(2 * np.pi * month / 12)
    month_cos = np.cos(2 * np.pi * month / 12)

    # Prepare input
    new_student = pd.DataFrame({
        "age": [age],
        "gender": [gender],
        "month_sin": [month_sin],
        "month_cos": [month_cos]
    })

    # Predict probability
    probability = model.predict_proba(new_student)[0, 1]

    # Apply 50% threshold
    prediction = (
        "respiratory-related"
        if probability >= 0.50
        else "other"
    )

    return probability, prediction


# Test the function
probability, prediction = predict_respiratory(21, "F", 6)

print("Respiratory probability:", round(probability * 100, 2), "%")
print("Predicted class:", prediction)

In [ ]:
!pip install streamlit

In [ ]:
from pathlib import Path

app_code = '''
import streamlit as st
import pandas as pd
import numpy as np
import joblib

# Load the trained Random Forest model
model_package = joblib.load(
    "capstone_outputs/respiratory_random_forest_model.joblib"
)

model = model_package["model"]

# Page configuration
st.set_page_config(
    page_title="Respiratory Illness Prediction",
    page_icon="🫁",
    layout="centered"
)

# Title
st.title("Respiratory Illness Prediction")
st.write(
    "Predict the estimated probability of a respiratory-related "
    "health-unit visit among university students."
)

st.divider()

# User inputs
age = st.number_input(
    "Age",
    min_value=15,
    max_value=100,
    value=20,
    step=1
)

gender = st.selectbox(
    "Gender",
    ["F", "M"]
)

month = st.selectbox(
    "Month",
    list(range(1, 13)),
    index=0
)

# Month names
month_names = {
    1: "January",
    2: "February",
    3: "March",
    4: "April",
    5: "May",
    6: "June",
    7: "July",
    8: "August",
    9: "September",
    10: "October",
    11: "November",
    12: "December"
}

st.write(f"Selected month: **{month_names[month]}**")

# Prediction
if st.button("Predict", type="primary"):

    # Create seasonal features
    month_sin = np.sin(2 * np.pi * month / 12)
    month_cos = np.cos(2 * np.pi * month / 12)

    # Prepare input
    new_student = pd.DataFrame({
        "age": [age],
        "gender": [gender],
        "month_sin": [month_sin],
        "month_cos": [month_cos]
    })

    # Predict probability
    probability = model.predict_proba(new_student)[0, 1]

    # Classification threshold
    if probability >= 0.50:
        prediction = "Respiratory-related"
    else:
        prediction = "Other"

    st.divider()

    st.subheader("Prediction Result")

    st.metric(
        "Respiratory-related probability",
        f"{probability * 100:.2f}%"
    )

    if prediction == "Respiratory-related":
        st.warning(
            "The model predicts a respiratory-related health-unit visit."
        )
    else:
        st.info(
            "The model predicts a non-respiratory health-unit visit."
        )

    st.write(f"**Predicted class:** {prediction}")

    st.caption(
        "This is a machine-learning prediction based on patterns "
        "in the training data and is not a medical diagnosis."
    )
'''

Path("app.py").write_text(app_code, encoding="utf-8")

print("app.py created successfully!")
print(Path("app.py").resolve())

In [ ]:
from pathlib import Path

model_path = Path("capstone_outputs/respiratory_random_forest_model.joblib")

print("Model exists:", model_path.exists())
print("Model path:", model_path.resolve())

In [ ]:
!streamlit run app.py

In [ ]:
from pathlib import Path
import joblib

model_path = Path("capstone_outputs/respiratory_random_forest_model.joblib")

print("File exists:", model_path.exists())
print("File size:", model_path.stat().st_size / (1024 * 1024), "MB")

test_model = joblib.load(model_path)

print("Model loaded successfully!")
print(type(test_model))